In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

In [2]:

df = pd.read_csv("../data/processed/elevator_clean.csv")

### we will train Random Forest with two types of spliting one is Random Split and another one is temporal split to check n prove data leakage problem

In [3]:
features = [
    "revolutions",
    "humidity",
    "x1",
    "x2",
    "x3",
    "x4",
    "x5"
]

X = df[features]
y = df["vibration"]

total_rows = len(df)

In [4]:
vibration = pd.Series(y)

lags = [1,4,16,64]

for lag in lags:

    correlation = vibration.autocorr(lag) # it gives the correlation between the series and its lagged version

    print(f"Lag {lag} : {correlation:.4f}")

Lag 1 : 0.3297
Lag 4 : 0.8589
Lag 16 : 0.9043
Lag 64 : 0.8865


In [5]:
import pandas as pd

# Create a sample time series
data = [10, 12, 14, 15, 17, 20, 22]
series = pd.Series(data)

# Compute autocorrelation for lag=1
correlation = series.autocorr(lag=1)
print(f"Autocorrelation at lag 1: {correlation}")


Autocorrelation at lag 1: 0.9868914563819992


In [6]:
df.head()

,Unnamed: 0,ID,revolutions,humidity,vibration,x1,x2,x3,x4,x5,Hour_Bin
0,0,1,93.744,73.999,18.0,167.743,19.745,1.266828,8787.937536,5475.852001,1
1,1,2,93.740,73.999,18.0,167.739,19.741,1.266774,8787.187600,5475.852001,1
2,2,3,93.736,73.998,18.0,167.734,19.738,1.266737,8786.437696,5475.704004,1
3,3,4,93.732,73.998,18.0,167.730,19.734,1.266683,8785.687824,5475.704004,1
4,4,5,93.729,73.998,18.0,167.727,19.731,1.266642,8785.125441,5475.704004,1


In [7]:
# Setting up the Random Forest Regressor model with specified hyperparameters
rf = RandomForestRegressor(

    n_estimators=100,
    max_depth=10,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1

)

#### 1 : Random Split

In [8]:
print("\nRandom Split")

np.random.seed(42)

indices = np.arange(total_rows)

np.random.shuffle(indices) 

split = int(total_rows*0.80)

train_index = indices[:split]

test_index = indices[split:] # we arranged the indices in a order then we shuffled them and then splited them into train and test indices


Random Split


In [9]:
rf.fit(X.iloc[train_index], y.iloc[train_index])

pred_random = rf.predict(X.iloc[test_index])

In [10]:
# we will calculate mae mse and r2 score for the random split
mae_random = mean_absolute_error(
    y.iloc[test_index],
    pred_random
)
rmse = np.sqrt(mean_squared_error(
    y.iloc[test_index],
    pred_random ))

r2_random = r2_score(
    y.iloc[test_index],
    pred_random
)

print("MAE :",mae_random)
print("RMSE :",rmse)
print("R2 :",r2_random)

MAE : 11.099186200207187
RMSE : 17.675315997941475
R2 : 0.4724997346241515


#### 2 : Temporal Split

In [11]:
print("\nTemporal Split")

split = int(total_rows*0.80)

X_train = X.iloc[:split]

X_test = X.iloc[split:]

y_train = y.iloc[:split]

y_test = y.iloc[split:]


Temporal Split


In [12]:
rf.fit(X_train,y_train)

pred_temporal = rf.predict(X_test)

In [13]:
mae_temporal = mean_absolute_error(
    y_test,
    pred_temporal
)
rmse_temporal = np.sqrt(mean_squared_error(
    y_test,
    pred_temporal
))
r2_temporal = r2_score(
    y_test,
    pred_temporal
)
print("MAE :",mae_temporal)
print("RMSE :",rmse_temporal)
print("R2 :",r2_temporal)

MAE : 15.37712440833846
RMSE : 16.705373932289728
R2 : -0.3066297179981303


In [14]:
# 1. Combine features and importances into a list of pairs
feature_pairs = list(zip(features, rf.feature_importances_))

# 2. Sort the list from highest importance to lowest
feature_pairs.sort(key=lambda item: item[1], reverse=True)

print("\nFeature Importances:")

# 3. Loop through and print each feature with its percentage
for name, importance in feature_pairs:
    percentage = importance * 100
    
    # Draw the bar (1 block per 2% of importance)
    bar_length = int(percentage / 2)
    bar = "█" * bar_length
    
    print(f"  {name}: {percentage:.1f}% {bar}")

# 4. explanation note
print("\nNote: x1 and x2 are the most important because they include revolutions.")
# Random Forest Experiments Done


Feature Importances:
  x2: 30.5% ███████████████
  x1: 22.1% ███████████
  humidity: 19.1% █████████
  x5: 18.6% █████████
  x3: 4.1% ██
  revolutions: 3.2% █
  x4: 2.4% █

Note: x1 and x2 are the most important because they include revolutions.


#####
We compared two evaluation methods using the same Random Forest model. In the random split, the model achieved an R² of 0.47 because similar sensor readings were present in both the training and testing sets, leading to data leakage. In the temporal split, where the model was trained only on past data and tested on future data, the R² dropped to -0.31. This revealed that the earlier result was overly optimistic and that the model could not generalize to unseen future data. Therefore, all subsequent experiments use temporal splitting to provide a realistic evaluation. 
#####